# Load data

In [2]:
import pandas as pd
df = df = pd.read_csv("transactions.csv")
print(df.head())

                narration        mode    type    category  subcategory  \
0    FD interest credited        NEFT  Credit  investment  fd_interest   
1               FD INT CR        NEFT  Credit  investment  fd_interest   
2  Fixed deposit interest        NEFT  Credit  investment  fd_interest   
3              FD booking  NETBANKING   Debit  investment   fd_booking   
4          Open FD online  NETBANKING   Debit  investment   fd_booking   

   Unnamed: 5  Unnamed: 6  Unnamed: 7  
0         NaN         NaN         NaN  
1         NaN         NaN         NaN  
2         NaN         NaN         NaN  
3         NaN         NaN         NaN  
4         NaN         NaN         NaN  


# Create input_text

In [3]:
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

## Lowercase

In [4]:
df["input_text"] = df["input_text"].str.lower()

# Create label (Ground Truth)

In [5]:
df["label"] = df["category"] + "/" + df["subcategory"]

In [6]:
print(df[["input_text", "label"]].head())

                                          input_text                   label
0  narration: fd interest credited | mode: neft |...  investment/fd_interest
1   narration: fd int cr | mode: neft | type: credit  investment/fd_interest
2  narration: fixed deposit interest | mode: neft...  investment/fd_interest
3  narration: fd booking | mode: netbanking | typ...   investment/fd_booking
4  narration: open fd online | mode: netbanking |...   investment/fd_booking


# Check label distribution

In [7]:
print(df["label"].value_counts())

label
investment/fd_interest    3
investment/fd_booking     2
investment/mutual_fund    2
expense/food              2
expense/shopping          2
income/salary             2
Name: count, dtype: int64


# Define X and y

In [8]:
X = df["input_text"]   
y = df["label"]       

# Train/Test Split

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # reproducibility
)

# Model Pipeline

In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=1000))
])

## Training

In [11]:
model.fit(X_train, y_train)

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


## Prediction

In [12]:
y_pred = model.predict(X_test)

In [13]:
from sklearn.metrics import f1_score, fbeta_score, classification_report

# F1 Score (balanced)
f1 = f1_score(y_test, y_pred, average='weighted')

# F2 Score (recall-focused)
f2 = fbeta_score(y_test, y_pred, beta=2, average='weighted')

print("F1 Score:", round(f1, 3))
print("F2 Score:", round(f2, 3))

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

F1 Score: 0.222
F2 Score: 0.278

Detailed Report:

                        precision    recall  f1-score   support

          expense/food       0.00      0.00      0.00         0
      expense/shopping       0.00      0.00      0.00         1
         income/salary       0.00      0.00      0.00         1
investment/fd_interest       0.50      1.00      0.67         1

              accuracy                           0.33         3
             macro avg       0.12      0.25      0.17         3
          weighted avg       0.17      0.33      0.22         3



c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

## Accuracy

In [14]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.3333333333333333


In [15]:
print(df[["narration", "label"]])

                 narration                   label
0     FD interest credited  investment/fd_interest
1                FD INT CR  investment/fd_interest
2   Fixed deposit interest  investment/fd_interest
3               FD booking   investment/fd_booking
4           Open FD online   investment/fd_booking
5                SIP debit  investment/mutual_fund
6       MF SIP installment  investment/mutual_fund
7           Swiggy payment            expense/food
8             Zomato order            expense/food
9          Amazon purchase        expense/shopping
10          Flipkart order        expense/shopping
11         Salary credited           income/salary
12          Monthly salary           income/salary


In [16]:
print(df["label"].value_counts())

label
investment/fd_interest    3
investment/fd_booking     2
investment/mutual_fund    2
expense/food              2
expense/shopping          2
income/salary             2
Name: count, dtype: int64


In [17]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[0 0 0 0]
 [1 0 0 0]
 [0 0 0 1]
 [0 0 0 1]]


In [18]:
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()

for i, label in enumerate(classifier.classes_):
    top_features = sorted(
        zip(classifier.coef_[i], feature_names),
        reverse=True
    )[:5]
    
    print(f"\nTop words for {label}:")
    for coef, word in top_features:
        print(word)


Top words for expense/food:
upi
zomato
swiggy
payment
order

Top words for expense/shopping:
flipkart
order
upi
debit
type

Top words for income/salary:
salary
monthly
neft
credit
type

Top words for investment/fd_booking:
netbanking
fd
booking
open
online

Top words for investment/fd_interest:
neft
credit
int
cr
interest

Top words for investment/mutual_fund:
sip
auto_debit
mf
installment
debit


## Debug

In [19]:
for i in range(len(X_test)):
    print("Input:", X_test.iloc[i])
    print("Actual:", y_test.iloc[i])
    print("Predicted:", y_pred[i])
    print("-----")

Input: narration: salary credited | mode: neft | type: credit
Actual: income/salary
Predicted: investment/fd_interest
-----
Input: narration: amazon purchase | mode: upi | type: debit
Actual: expense/shopping
Predicted: expense/food
-----
Input: narration: fd interest credited | mode: neft | type: credit
Actual: investment/fd_interest
Predicted: investment/fd_interest
-----


## Confidence

In [20]:
probs = model.predict_proba(X_test)
for i in range(len(X_test)):
    print("Prediction:", model.classes_[probs[i].argmax()])
    print("Confidence:", round(probs[i].max(), 2))

Prediction: investment/fd_interest
Confidence: 0.26
Prediction: expense/food
Confidence: 0.28
Prediction: investment/fd_interest
Confidence: 0.33


In [21]:
def confidence_level(conf):
    if conf > 0.75:
        return "High"
    elif conf > 0.5:
        return "Medium"
    else:
        return "Low"

In [22]:
from sklearn.metrics import classification_report, f1_score

print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

F1 Score: 0.2222222222222222

Detailed Report:

                        precision    recall  f1-score   support

          expense/food       0.00      0.00      0.00         0
      expense/shopping       0.00      0.00      0.00         1
         income/salary       0.00      0.00      0.00         1
investment/fd_interest       0.50      1.00      0.67         1

              accuracy                           0.33         3
             macro avg       0.12      0.25      0.17         3
          weighted avg       0.17      0.33      0.22         3



c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modif

# Output

In [23]:
results = []
for i in range(len(df)):
    input_text = df["input_text"].iloc[i]
    probs = model.predict_proba([input_text])[0]
    pred_label = model.classes_[probs.argmax()]
    confidence = probs.max()
    category, subcategory = pred_label.split("/", 1)
    results.append({
    "narration": df["narration"].iloc[i],
    "mode": df["mode"].iloc[i],
    "type": df["type"].iloc[i],
    "ground_truth": f"{category}/{subcategory}",
    "confidence": round(confidence, 2),
    "confidence_percent": f"{round(confidence*100, 1)}%",
    "confidence_level": confidence_level(confidence)
})
result_df = pd.DataFrame(results)
print(result_df.to_string())

                 narration        mode    type            ground_truth  confidence confidence_percent confidence_level
0     FD interest credited        NEFT  Credit  investment/fd_interest        0.33              32.8%              Low
1                FD INT CR        NEFT  Credit  investment/fd_interest        0.36              35.6%              Low
2   Fixed deposit interest        NEFT  Credit  investment/fd_interest        0.37              37.2%              Low
3               FD booking  NETBANKING   Debit   investment/fd_booking        0.37              36.8%              Low
4           Open FD online  NETBANKING   Debit   investment/fd_booking        0.38              37.7%              Low
5                SIP debit  AUTO_DEBIT   Debit  investment/mutual_fund        0.37              37.0%              Low
6       MF SIP installment  AUTO_DEBIT   Debit  investment/mutual_fund        0.40              39.8%              Low
7           Swiggy payment         UPI   Debit  